# Module 11 — Chunks + Qdrant (similarité sémantique)

Compléter le BM25 (module 10) par des **embeddings** sur des **chunks citables**.

**Prérequis** : `docker compose up -d qdrant`, `QDRANT_URL` dans `.env`, corpus indexé OpenSearch (`uv run presslake index`).

Tuto pas à pas : [`docs/modules/11-chunking-qdrant.md`](../docs/modules/11-chunking-qdrant.md)

## Étape 1 — Santé Qdrant

Même logique qu'OpenSearch au module 10 : vérifier que le service répond avant d'écrire.

In [1]:
from presslake.vector.client import get_qdrant_client
from presslake.vector.collection import COLLECTION_CHUNKS, count_points

client = get_qdrant_client()
print("collections:", [c.name for c in client.get_collections().collections])
print(f"points dans {COLLECTION_CHUNKS}:", count_points(client))

collections: ['presslake-chunks']
points dans presslake-chunks: 8


## Étape 2 — Chunking pur (sans ML)

On découpe le texte **avant** d'appeler un modèle :
- `max_chars=800` : taille raisonnable pour un contexte LLM
- `overlap=100` : évite de couper une idée entre deux chunks
- `char_start` / `char_end` : offsets pour citer le silver original

In [2]:
from presslake.chunk.split import chunk_text

sample = (
    "Au Népal, une crue meurtrière a dévasté des villages. "
    "Les glaciologues s'interrogent sur l'ampleur de la catastrophe. "
    * 8
)

pieces = chunk_text(sample, max_chars=120, overlap=30)
print(f"{len(sample)} car. → {len(pieces)} chunk(s)\n")
for p in pieces:
    print(f"[{p['chunk_index']}] {p['char_start']}–{p['char_end']} | {p['text'][:80]}…")

944 car. → 11 chunk(s)

[0] 0–117 | Au Népal, une crue meurtrière a dévasté des villages. Les glaciologues s'interro…
[1] 87–206 | r l'ampleur de la catastrophe. Au Népal, une crue meurtrière a dévasté des villa…
[2] 176–293 | glaciologues s'interrogent sur l'ampleur de la catastrophe. Au Népal, une crue m…
[3] 263–372 | re a dévasté des villages. Les glaciologues s'interrogent sur l'ampleur de la ca…
[4] 342–458 | atastrophe. Au Népal, une crue meurtrière a dévasté des villages. Les glaciologu…
[5] 428–542 | nterrogent sur l'ampleur de la catastrophe. Au Népal, une crue meurtrière a déva…
[6] 512–629 | des villages. Les glaciologues s'interrogent sur l'ampleur de la catastrophe. Au…
[7] 599–717 | une crue meurtrière a dévasté des villages. Les glaciologues s'interrogent sur l…
[8] 687–806 | r de la catastrophe. Au Népal, une crue meurtrière a dévasté des villages. Les g…
[9] 776–883 | es s'interrogent sur l'ampleur de la catastrophe. Au Népal, une crue meurtrière …
[10] 853–943 | re a

## Étape 3 — Chunks depuis un silver réel

`silver_to_chunks` ajoute les metadata de **citation** (hash, langue, URI silver).

In [3]:
from presslake.chunk.envelope import silver_to_chunks
from presslake.storage.postgres import get_connection
from presslake.storage.s3 import get_json_object, get_s3_client, parse_s3_uri

with get_connection() as conn:
    row = conn.execute(
        """
        SELECT silver_s3_uri, title
        FROM articles
        WHERE silver_s3_uri IS NOT NULL AND status IN ('indexed', 'embedded')
        ORDER BY fetched_at DESC
        LIMIT 1
        """
    ).fetchone()

if not row:
    raise RuntimeError("Aucun silver — lance `presslake index` d'abord.")

silver_uri, title = row
bucket, key = parse_s3_uri(silver_uri)
silver = get_json_object(get_s3_client(), bucket, key)
chunks = silver_to_chunks(silver, silver_s3_uri=silver_uri)

print(f"Article : {title[:70]}")
print(f"Chunks  : {len(chunks)}")
print(f"Exemple : {chunks[0]['chunk_id']}")
print(f"Citation: {chunks[0]['silver_s3_uri']} [{chunks[0]['char_start']}:{chunks[0]['char_end']}]")

Article : Terrance Tao explains 6 essential mathematical concepts [video]
Chunks  : 1
Exemple : aed34259e46147002c09b749572ec2359551b79609948a27e7f52a0e3df2a5d4:0
Citation: s3://presslake/silver/source=hnrss/dt=2026-08-30/aed34259e46147002c09b749572ec2359551b79609948a27e7f52a0e3df2a5d4.json [0:141]


## Étape 4 — Embeddings (premier appel = téléchargement ONNX)

Modèle : `paraphrase-multilingual-MiniLM-L12-v2` (384 dimensions, multilingue ADR 0003).

Query et passages utilisent le **même** encodeur (pas de préfixe E5).

In [4]:
from presslake.vector.config import VECTOR_SIZE
from presslake.vector.embed import embed_passages, embed_query

vec_q = embed_query("catastrophe himalayenne")
vec_p = embed_passages(["inondations au Népal dans l'Himalaya"])[0]

print(f"dimension : {len(vec_q)} (attendu {VECTOR_SIZE})")
print(f"extrait query : {vec_q[:3]}…")

/home/anthony-marais/Documents/data_project/src/presslake/vector/embed.py:13: UserWarning: The model sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 now uses mean pooling instead of CLS embedding. In order to preserve the previous behaviour, consider either pinning fastembed version to 0.5.1 or using `add_custom_model` functionality.
  return TextEmbedding(model_name=EMBEDDING_MODEL)


dimension : 384 (attendu 384)
extrait query : [-0.031224568684895832, 0.09862179226345485, 0.043923272026909724]…


## Étape 5 — Embeder le corpus

```bash
uv run presslake embed --limit 10   # test rapide
uv run presslake embed              # tout le corpus indexed
```

Relance la cellule 1 pour voir le nombre de points augmenter.

## Étape 6 — Helper d'affichage + cas de test

In [5]:
from presslake.vector.client import get_qdrant_client
from presslake.vector.collection import search_similar
from presslake.vector.embed import embed_query
from presslake.search.client import get_opensearch_client
from presslake.search.index import search_articles


def show_similar(query: str, *, limit: int = 5, lang: str | None = None, note: str = "") -> list[dict]:
  vector = embed_query(query)
  hits = search_similar(get_qdrant_client(), vector, limit=limit, lang=lang)
  lang_hint = f" (lang={lang})" if lang else ""
  print(f"Requête : {query!r}{lang_hint}")
  if note:
    print(f"Note    : {note}")
  print(f"Résultats : {len(hits)}\n")
  if not hits:
    print("→ Aucun hit (collection vide ? lance `presslake embed`).")
    return hits
  for i, hit in enumerate(hits, 1):
    title = (hit.get("title") or "(sans titre)")[:60]
    print(f"{i}. [{hit['score']:.3f}] {hit['feed_id']} | chunk {hit.get('chunk_index')} | {title}")
    print(f"   … {(hit.get('text') or '')[:100]}…")
    print(f"   silver: {hit.get('silver_s3_uri')}")
  print()
  return hits

### Cas A — Paraphrase (complète le cas E du module 10)

**Attendu** : chunks sur le Népal **sans** écrire le mot exact dans la requête.

In [6]:
hits_para = show_similar(
    "catastrophe himalayenne crue meurtrière",
    note="Paraphrase — similarité sémantique",
)

Requête : 'catastrophe himalayenne crue meurtrière'
Note    : Paraphrase — similarité sémantique
Résultats : 5

1. [0.467] france24 | chunk 1 | Crise au Népal : onde de choc régionale ?
   … La catastrophe climatique d'aujourd'hui peut-elle dégénérer en une catastrophe humanitaire ? Doit-on…
   silver: s3://presslake/silver/source=france24/dt=2026-08-31/feb162e21ec643072a8da0daf1034cf458c5479cfef5ab5ddd9d8d75d6c7a0b1.json
2. [0.323] france24 | chunk 0 | Crise au Népal : onde de choc régionale ?
   … Crise au Népal : onde de choc régionale ?
Pour afficher ce contenu YouTube, il est nécessaire d'auto…
   silver: s3://presslake/silver/source=france24/dt=2026-08-31/feb162e21ec643072a8da0daf1034cf458c5479cfef5ab5ddd9d8d75d6c7a0b1.json
3. [0.291] france24 | chunk 5 | 🔴 L'ancien Premier ministre Édouard Balladur est mort à l'âg
   … contrats d'armement avec le Pakistan. Il est relaxé par la Cour de justice de la République en 2021,…
   silver: s3://presslake/silver/source=france24/dt=2026-08-

### Cas B — Mot exact

**Attendu** : chunks contenant « Népal » toujours pertinents (lexical + sémantique).

In [7]:
hits_nepal = show_similar("Népal inondations glaciologues")

Requête : 'Népal inondations glaciologues'
Résultats : 5

1. [0.522] france24 | chunk 0 | Crise au Népal : onde de choc régionale ?
   … Crise au Népal : onde de choc régionale ?
Pour afficher ce contenu YouTube, il est nécessaire d'auto…
   silver: s3://presslake/silver/source=france24/dt=2026-08-31/feb162e21ec643072a8da0daf1034cf458c5479cfef5ab5ddd9d8d75d6c7a0b1.json
2. [0.299] france24 | chunk 1 | Crise au Népal : onde de choc régionale ?
   … La catastrophe climatique d'aujourd'hui peut-elle dégénérer en une catastrophe humanitaire ? Doit-on…
   silver: s3://presslake/silver/source=france24/dt=2026-08-31/feb162e21ec643072a8da0daf1034cf458c5479cfef5ab5ddd9d8d75d6c7a0b1.json
3. [0.140] france24 | chunk 3 | 🔴 L'ancien Premier ministre Édouard Balladur est mort à l'âg
   … la droite perd les législatives. Elle tient sa revanche cinq ans plus tard aux législatives de 1993,…
   silver: s3://presslake/silver/source=france24/dt=2026-08-31/fb78ed72468c45b3895375890d347f6cf7c19e1f9ebe8157832

### Cas C — Cross-lingue

**Attendu** : requête EN peut matcher des articles FR (modèle multilingue MiniLM).

In [8]:
hits_en = show_similar("Nepal flood disaster Himalayan", note="Requête anglaise, corpus mixte")

Requête : 'Nepal flood disaster Himalayan'
Note    : Requête anglaise, corpus mixte
Résultats : 5

1. [0.542] france24 | chunk 0 | Crise au Népal : onde de choc régionale ?
   … Crise au Népal : onde de choc régionale ?
Pour afficher ce contenu YouTube, il est nécessaire d'auto…
   silver: s3://presslake/silver/source=france24/dt=2026-08-31/feb162e21ec643072a8da0daf1034cf458c5479cfef5ab5ddd9d8d75d6c7a0b1.json
2. [0.296] france24 | chunk 1 | Crise au Népal : onde de choc régionale ?
   … La catastrophe climatique d'aujourd'hui peut-elle dégénérer en une catastrophe humanitaire ? Doit-on…
   silver: s3://presslake/silver/source=france24/dt=2026-08-31/feb162e21ec643072a8da0daf1034cf458c5479cfef5ab5ddd9d8d75d6c7a0b1.json
3. [0.123] france24 | chunk 0 | 🔴 L'ancien Premier ministre Édouard Balladur est mort à l'âg
   … Mort de l'ancien Premier ministre Édouard Balladur à 97 ans
Édouard Balladur, Premier ministre entre…
   silver: s3://presslake/silver/source=france24/dt=2026-08-31/fb78ed7246

### Cas D — BM25 vs similar (justifie le module 12)

Compare la **même paraphrase** : OpenSearch (mots) vs Qdrant (sens).

In [9]:
query = "catastrophe himalayenne crue meurtrière"

bm25 = search_articles(get_opensearch_client(), query, limit=3)
sem = show_similar(query, limit=3)

print("--- Comparaison ---")
print(f"BM25 (search)  : {len(bm25)} hits", end="")
if bm25:
    print(f", top = {bm25[0]['title'][:50]!r}")
else:
    print()
print(f"Qdrant (similar): {len(sem)} hits", end="")
if sem:
    print(f", top score = {sem[0]['score']:.3f}")
else:
    print()

Requête : 'catastrophe himalayenne crue meurtrière'
Résultats : 3

1. [0.467] france24 | chunk 1 | Crise au Népal : onde de choc régionale ?
   … La catastrophe climatique d'aujourd'hui peut-elle dégénérer en une catastrophe humanitaire ? Doit-on…
   silver: s3://presslake/silver/source=france24/dt=2026-08-31/feb162e21ec643072a8da0daf1034cf458c5479cfef5ab5ddd9d8d75d6c7a0b1.json
2. [0.323] france24 | chunk 0 | Crise au Népal : onde de choc régionale ?
   … Crise au Népal : onde de choc régionale ?
Pour afficher ce contenu YouTube, il est nécessaire d'auto…
   silver: s3://presslake/silver/source=france24/dt=2026-08-31/feb162e21ec643072a8da0daf1034cf458c5479cfef5ab5ddd9d8d75d6c7a0b1.json
3. [0.291] france24 | chunk 5 | 🔴 L'ancien Premier ministre Édouard Balladur est mort à l'âg
   … contrats d'armement avec le Pakistan. Il est relaxé par la Cour de justice de la République en 2021,…
   silver: s3://presslake/silver/source=france24/dt=2026-08-31/fb78ed72468c45b3895375890d347f6cf7c19e1f9e

## Étape 7 — Synthèse

| Cas | Type | Moteur |
|---|---|---|
| A | Paraphrase | Qdrant ✅ |
| B | Mot exact | Les deux ✅ |
| C | Cross-lingue | Qdrant ✅ (MiniLM multilingue) |
| D | Comparaison | BM25 + similar → hybride module 12 |

**Score cosine** : 0–1, plus haut = plus similaire *à cette requête*.

CLI : `uv run presslake similar "catastrophe himalayenne"`